# **1. Rozszerzenie modelu**
Dodano nowy wymiar DimCountry, który umożliwia analizę sprzedaży według krajów.

In [14]:
import pandas as pd

df = pd.read_csv("Online_Retail.csv", encoding='ISO-8859-1')

# czyszczenie (skrócone)
df = df.dropna(subset=["CustomerID"])
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

# DimCountry
dim_country = df[["Country"]].drop_duplicates().copy()
dim_country["country_key"] = range(1, len(dim_country) + 1)

/tmp/ipykernel_9335/216106043.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


# **2. Rozszerzona tabela faktów**

In [15]:
fact = df.merge(dim_country, on="Country")

fact_sales = fact[[
    "country_key",
    "Quantity",
    "Revenue"
]]

Zmiana danych (np. Country) jest wykrywana na podstawie różnicy wartości dla tego samego CustomerID, co skutkuje utworzeniem nowego rekordu.

#**3.SCD Typ 2**

Zaimplementowano SCD typu 2 dla wymiaru DimCustomer.

In [16]:
dim_customer = df[["CustomerID", "Country"]].drop_duplicates().copy()
dim_customer["customer_key"] = range(1, len(dim_customer) + 1)

dim_customer["valid_from"] = pd.Timestamp("2020-01-01")
dim_customer["valid_to"] = pd.NaT
dim_customer["is_current"] = True

#**4. Analiza projektu**
Ziarno na poziomie pozycji faktury zostało wybrane, ponieważ zapewnia największą szczegółowość danych i elastyczność analiz.

Kompromisem jest większy rozmiar danych, jednak umożliwia to dokładniejsze analizy.

Model wspiera analizy biznesowe poprzez możliwość analizy sprzedaży według produktów, klientów, czasu oraz krajów.


WYNIKI

In [17]:
display(dim_country.head())
display(fact_sales.head())

,Country,country_key
0,United Kingdom,1
26,France,2
197,Australia,3
385,Netherlands,4
1109,Germany,5


,country_key,Quantity,Revenue
0,1,6,15.30
1,1,6,20.34
2,1,8,22.00
3,1,6,20.34
4,1,6,20.34
